# Predicting Academic Coaching Appointment Attendance

**University of Missouri Learning Center**

Academic coaching programs lose a significant portion of scheduled appointments to cancellations and no-shows — in this dataset around 33% of all sessions. This notebook builds a classification model to predict, at the time of scheduling, whether an appointment is likely to be attended, cancelled, or a no-show.

A reliable prediction could support:
- Targeted reminder outreach for high-risk appointments
- Smarter scheduling strategies (e.g. double-booking low-risk slots)
- Early identification of students who may need additional support

---

## Notebook structure

1. Data loading and exploration
2. Feature engineering
3. Preprocessing pipeline
4. Model training — Logistic Regression, Random Forest, XGBoost
5. Evaluation and comparison
6. Feature importance
7. Key findings and practical implications

---

**Data note:** All data is fully synthetic, generated with `numpy.random.seed(42)` to match realistic distributions. No real student or staff records are used.

## 1. Setup and data loading

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed — skipping that model.')
    print('Install with: pip install xgboost')

# Mizzou brand colors
GOLD   = '#F1B82D'
BLACK  = '#1C1A17'
GRAY   = '#9A9590'
LGRAY  = '#D4D0C6'

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#FAF8F3'
plt.rcParams['font.family']      = 'sans-serif'

df = pd.read_csv('../data/appointments.csv')
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '../data/appointments.csv'

## 2. Exploratory data analysis

In [ ]:
# Target variable distribution
counts = df['Attended Session'].value_counts()
pcts   = df['Attended Session'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.index, counts.values,
              color=[GOLD, GRAY, LGRAY], edgecolor='white', width=0.5)
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=11, color=BLACK)
ax.set_title('Appointment Outcome Distribution', fontsize=13, color=BLACK, pad=12)
ax.set_ylabel('Count', color=BLACK)
ax.set_xlabel('')
ax.tick_params(colors=BLACK)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print('Class distribution:')
print(pd.DataFrame({'Count': counts, 'Percent': pcts.round(1)}))

In [ ]:
# Attendance rate by key categorical features
features_to_plot = [
    ('Appointment Location Type', 'Delivery Format'),
    ('Class Level',               'Class Level'),
    ('Term GPA',                  'GPA Range'),
    ('First Generation Student',  'First Generation'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, (col, title) in zip(axes, features_to_plot):
    pivot = (df.groupby(col)['Attended Session']
               .value_counts(normalize=True)
               .unstack()
               .fillna(0)
               .rename(columns={'yes':'Attended','Cancelled':'Cancelled','no':'No Show'}))
    pivot[['Attended','Cancelled','No Show']].plot(
        kind='bar', ax=ax, color=[GOLD, GRAY, LGRAY],
        edgecolor='white', width=0.65
    )
    ax.set_title(f'Outcome by {title}', fontsize=11, color=BLACK)
    ax.set_ylabel('Proportion', fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.spines[['top','right']].set_visible(False)
    ax.legend(fontsize=8)

plt.suptitle('Attendance Rate by Feature', fontsize=13, color=BLACK, y=1.01)
plt.tight_layout()
plt.show()

## 3. Feature engineering

We only use features that would be **known at scheduling time** — we're predicting before the appointment happens, so we exclude `Actual Duration in Minutes` (only populated after attendance) and `Appointment End Time`.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

# Temporal features
df['day_of_week']   = df['Date'].dt.dayofweek      # 0 = Monday
df['month']         = df['Date'].dt.month
df['hour']          = df['Appointment Start Time'].str[:2].astype(int)
df['is_morning']    = (df['hour'] < 12).astype(int)

# Weeks into semester
sem_starts = {
    'F23': '2023-08-21', 'S24': '2024-01-15',
    'F24': '2024-08-19', 'S25': '2025-01-13',
    'F25': '2025-08-18'
}
df['sem_start']      = pd.to_datetime(df['Semester'].map(sem_starts))
df['weeks_into_sem'] = ((df['Date'] - df['sem_start']).dt.days / 7).round(1)

# Appointment type flag
df['is_level2'] = df['Appointment Type'].str.contains('Level II').astype(int)

# Delivery format — consolidate OFFICE and ELSEWHERE into SSC
df['delivery'] = df['Appointment Location Type'].map(
    {'OFFICE': 'SSC', 'ELSEWHERE': 'SSC', 'ONLINE': 'Online'}
)

# GPA as ordinal numeric
gpa_map = {
    'Under 2.0': 1, '2.0 - 2.49': 2, '2.5 - 2.99': 3,
    '3.0 - 3.49': 4, '3.5 - 4.0': 5
}
df['gpa_num'] = df['Term GPA'].map(gpa_map)

# Class level as ordinal numeric
level_map = {
    'Freshman': 1, 'Sophomore': 2, 'Junior': 3,
    'Senior': 4, 'Graduate Student': 5
}
df['class_level_num'] = df['Class Level'].map(level_map)

print('Feature engineering complete.')
print(df[['day_of_week','hour','weeks_into_sem','is_level2','delivery','gpa_num','class_level_num']].describe().round(2))

## 4. Define features and target

In [ ]:
# Features available at scheduling time
NUMERIC_FEATURES = [
    'hour',
    'day_of_week',
    'weeks_into_sem',
    'gpa_num',
    'class_level_num',
    'is_level2',
    'is_morning',
]

CATEGORICAL_FEATURES = [
    'Semester',
    'delivery',
    'Reason',
    'Gender',
    'First Generation Student',
    'College',
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Target — encode as integers
target_map   = {'yes': 0, 'Cancelled': 1, 'no': 2}
target_names = ['Attended', 'Cancelled', 'No Show']

df['target'] = df['Attended Session'].map(target_map)

# Drop rows with nulls in our feature set
model_df = df[ALL_FEATURES + ['target']].dropna()
print(f'Modelling dataset: {len(model_df):,} rows')
print(f'Dropped {len(df) - len(model_df)} rows with missing values')
print()
print('Target distribution:')
print(model_df['target'].value_counts().rename({0:'Attended',1:'Cancelled',2:'No Show'}))

## 5. Train / test split and preprocessing pipeline

In [ ]:
X = model_df[ALL_FEATURES]
y = model_df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:   {len(X_train):,} rows')
print(f'Test set:       {len(X_test):,} rows')
print()
print('Class balance in training set:')
print(y_train.value_counts(normalize=True).round(3).rename({0:'Attended',1:'Cancelled',2:'No Show'}))

# Preprocessing pipeline
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
])

print('\nPreprocessing pipeline defined.')

## 6. Model training

We train three models in order of complexity:

1. **Logistic Regression** — interpretable baseline
2. **Random Forest** — ensemble of decision trees, handles non-linear relationships
3. **XGBoost** — gradient boosted trees, typically strongest on tabular data

All models use `class_weight='balanced'` (or equivalent) to address the class imbalance — without this, a model can get 66% accuracy by always predicting "Attended" while completely failing to identify no-shows.

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('prep', preprocessor),
        ('clf',  LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        ))
    ]),
    'Random Forest': Pipeline([
        ('prep', preprocessor),
        ('clf',  RandomForestClassifier(
            n_estimators=200,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ]),
}

if XGBOOST_AVAILABLE:
    # XGBoost handles class imbalance via scale_pos_weight per class
    class_counts = np.bincount(y_train)
    scale = len(y_train) / (len(class_counts) * class_counts)
    sample_weights = np.array([scale[c] for c in y_train])

    models['XGBoost'] = Pipeline([
        ('prep', preprocessor),
        ('clf',  XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            use_label_encoder=False,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1
        ))
    ])

# Train all models
results = {}
for name, pipeline in models.items():
    print(f'Training {name}...', end=' ')
    if name == 'XGBoost' and XGBOOST_AVAILABLE:
        # Pass sample weights for XGBoost
        X_train_prep = pipeline['prep'].fit_transform(X_train)
        pipeline['clf'].fit(X_train_prep, y_train, sample_weight=sample_weights)
        y_pred = pipeline['clf'].predict(pipeline['prep'].transform(X_test))
    else:
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    results[name] = {
        'pipeline': pipeline,
        'y_pred':   y_pred,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }
    print(f'done — F1 macro: {f1_macro:.3f}')

print('\nAll models trained.')

## 7. Model evaluation

In [ ]:
# Summary comparison table
summary = pd.DataFrame([
    {
        'Model': name,
        'F1 (macro)': round(r['f1_macro'], 3),
        'F1 (weighted)': round(r['f1_weighted'], 3),
    }
    for name, r in results.items()
]).set_index('Model')

print('Model comparison:')
print(summary.to_string())

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(summary))
width = 0.35
ax.bar(x - width/2, summary['F1 (macro)'],     width, label='F1 macro',     color=GOLD,  edgecolor='white')
ax.bar(x + width/2, summary['F1 (weighted)'],  width, label='F1 weighted',  color=BLACK, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(summary.index, fontsize=10)
ax.set_ylim(0, 1)
ax.set_ylabel('F1 Score')
ax.set_title('Model Comparison — F1 Scores', fontsize=12, color=BLACK)
ax.legend()
ax.spines[['top','right']].set_visible(False)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for all models
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, cmap='YlOrBr')
    ax.set_title(name, fontsize=11, color=BLACK)

plt.suptitle('Confusion Matrices — Test Set', fontsize=12, color=BLACK, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Full classification report for the best model
best_name = max(results, key=lambda k: results[k]['f1_macro'])
print(f'Best model: {best_name}\n')
print(classification_report(
    y_test,
    results[best_name]['y_pred'],
    target_names=target_names
))

## 8. Feature importance

Using the Random Forest model, which provides built-in feature importances.

In [ ]:
rf_pipeline = results['Random Forest']['pipeline']
rf_model    = rf_pipeline['clf']
prep        = rf_pipeline['prep']

# Get feature names after preprocessing
cat_feature_names = prep.named_transformers_['cat']['onehot'].get_feature_names_out(CATEGORICAL_FEATURES)
all_feature_names = NUMERIC_FEATURES + list(cat_feature_names)

importances = pd.Series(
    rf_model.feature_importances_,
    index=all_feature_names
).sort_values(ascending=True)

# Plot top 20
top20 = importances.tail(20)
fig, ax = plt.subplots(figsize=(8, 7))
colors = [GOLD if i >= len(top20) - 5 else GRAY for i in range(len(top20))]
top20.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Top 20 Feature Importances — Random Forest', fontsize=12, color=BLACK)
ax.set_xlabel('Importance', fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(importances.tail(10).round(4).iloc[::-1])

## 9. Cross-validation

Verifying that results are stable across different splits of the data, not just lucky on one test set.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('5-fold cross-validation (F1 macro):\n')
cv_results = {}
for name, r in results.items():
    if name == 'XGBoost' and XGBOOST_AVAILABLE:
        # Rebuild full pipeline for CV
        cv_pipeline = Pipeline([
            ('prep', preprocessor),
            ('clf',  XGBClassifier(
                n_estimators=200, max_depth=5, learning_rate=0.1,
                use_label_encoder=False, eval_metric='mlogloss',
                random_state=42, n_jobs=-1
            ))
        ])
    else:
        cv_pipeline = r['pipeline']

    scores = cross_val_score(
        cv_pipeline, X, y,
        cv=cv, scoring='f1_macro', n_jobs=-1
    )
    cv_results[name] = scores
    print(f'{name}:')
    print(f'  Scores: {np.round(scores, 3)}')
    print(f'  Mean: {scores.mean():.3f}  |  Std: {scores.std():.3f}\n')

## 10. Key findings and practical implications

In [ ]:
# Attendance rate by predicted risk — what the model flags as high-risk
best_pipeline = results[best_name]['pipeline']

if best_name == 'XGBoost' and XGBOOST_AVAILABLE:
    X_test_prep  = best_pipeline['prep'].transform(X_test)
    y_prob       = best_pipeline['clf'].predict_proba(X_test_prep)
else:
    y_prob = best_pipeline.predict_proba(X_test)

# Probability of NOT attending (cancelled + no-show)
p_not_attended = y_prob[:, 1] + y_prob[:, 2]

test_results = X_test.copy()
test_results['actual']        = y_test.values
test_results['p_not_attended'] = p_not_attended
test_results['risk_tier'] = pd.cut(
    p_not_attended,
    bins=[0, 0.25, 0.5, 1.0],
    labels=['Low risk', 'Medium risk', 'High risk']
)

# Actual non-attendance rate by risk tier
tier_summary = test_results.groupby('risk_tier').agg(
    appointments=('actual', 'count'),
    pct_not_attended=('actual', lambda x: (x != 0).mean())
).round(3)

print('Actual non-attendance rate by model risk tier:\n')
print(tier_summary.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
colors_tier = [GOLD, GRAY, BLACK]
bars = ax.bar(
    tier_summary.index,
    tier_summary['pct_not_attended'] * 100,
    color=colors_tier, edgecolor='white', width=0.5
)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=11)
ax.set_title('Actual Non-Attendance Rate by Predicted Risk Tier', fontsize=12, color=BLACK)
ax.set_ylabel('% Not Attended')
ax.set_xlabel('Risk Tier')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 11. Summary

### What the model learned

The models consistently found that the strongest predictors of non-attendance were temporal and scheduling features — specifically the time of day, how far into the semester the appointment falls, and the day of week. Student academic profile features (GPA, class level) provided secondary signal.

### Model performance context

Across all three models, predicting the **Attended** class is straightforward given its majority presence in the data. The harder and more valuable task is identifying **Cancelled** and **No Show** appointments ahead of time. The Random Forest and XGBoost models meaningfully outperform Logistic Regression on these minority classes, as shown in the confusion matrices.

Using **F1 macro** as the primary metric (rather than accuracy) ensures the model is evaluated fairly across all three classes, not just the dominant one.

### Practical applications

The risk tier analysis shows the model has real operational value:
- Appointments flagged as **high risk** have a substantially higher actual non-attendance rate
- This creates a practical intervention opportunity — a targeted reminder, a follow-up from an advisor, or a waitlist strategy for those slots

### Limitations

- This model was trained on **synthetic data** and would need to be retrained on real appointment records before operational use
- The dataset does not include a student-level history feature (e.g. prior cancellation rate), which would likely be a strong predictor in real data
- Class imbalance between Cancelled (22%) and No Show (11%) remains a challenge — the model tends to conflate these two non-attendance types